# Week 3: Control (Individual)

In Week 3, you will build on your estimation approach to design a strategy for influencing the system.

## Objective

Your goal is to design a control policy that uses the available information (observations and/or your estimator) to generate inputs \(u(t)\) that produce a desired behaviour.

There is **no single correct solution**. You are expected to propose, justify, implement, and evaluate your own approach.

## What does “control” mean here?

Depending on your design choices, your controller may aim to:

- stabilise part of the system dynamics,
- drive the system toward a desired state,
- track a target trajectory,
- regulate or suppress certain behaviours,
- shape the system response over time.

Your task is not simply to produce outputs, but to design a strategy that meaningfully interacts with the system dynamics.

## Tasks

You should:

1. **Define a control objective**  
   Clearly specify the behaviour you want to achieve and explain why it is meaningful.

2. **Design a control strategy**  
   Propose one or more methods for generating inputs based on observations and/or estimated latent representations. Where possible, compare different approaches and justify your final design choices.

3. **Implement your controller**  
   Integrate your control policy with the simulation framework developed in Week 1 and your estimation pipeline from Week 2.

4. **Test in closed loop**  
   Run experiments where your controller continuously interacts with the system over time.

5. **Evaluate performance**  
   Use plots and quantitative measures to analyse:
   - stability,
   - responsiveness,
   - control effort,
   - robustness to noise,
   - sensitivity to parameter choices.

6. **Iterate and refine**  
   Improve your approach based on observed behaviour and failed attempts.

## Guidance

- You are encouraged to reuse and adapt your estimation approach from Week 2.
- Simple, well-justified methods are preferable to complex but poorly understood ones.
- Inputs are clipped internally to the range \([0,1]\), so excessively large values may have no additional effect.
- A controller that partially works but is carefully analysed is preferable to a complex controller that is poorly understood.
- Keep clear records of:
  - what you tried,
  - what worked,
  - what failed,
  - and how your approach evolved.

## Demonstrator check-ins

During Week 3, you are expected to demonstrate steady progress on your control design.

The following sessions are **compulsory**:

- **Tuesday 2 June, 9–11am**  
  Initial control integration check:
  - interaction with the simulator,
  - closed-loop execution,
  - initial plots and debugging.

- **Friday 5 June, 11am–1pm**  
  Progress review:
  - presentation of your current method,
  - discussion of results,
  - comparison of different approaches explored so far.

These sessions are **not formally marked**, but they are intended to help you refine your approach and prepare for the final stage of the project.

## What you should be prepared to show

You should be able to demonstrate:

1. **A working closed-loop experiment**  
   A simulation where your controller actively influences the system behaviour over time.

2. **Performance plots**  
   For example:
   - system outputs over time,
   - comparison between uncontrolled and controlled behaviour,
   - evidence of stability or instability.

3. **Identified issues or limitations**  
   For example:
   - instability,
   - saturation of inputs,
   - slow or ineffective response,
   - sensitivity to noise.

4. **An attempted improvement**  
   A modification you introduced to address the issue, and the effect it had.

You should also be able to clearly explain:

- your control objective,
- what information your controller uses,
- how your control input \(u(t)\) is computed,
- why you believe your approach is reasonable.

The aim of these sessions is to support iteration, debugging, and critical evaluation — not to arrive immediately at a perfect solution.

## Connection to later work

Your control strategy will be evaluated and compared with others in Week 4.

You should begin thinking about:
- how to present your results clearly,
- how to compare different approaches fairly,
- and how to justify your design decisions and conclusions.

## Accessing the simulator

By Week 3, you now have access to the simulator interface that allows you to interact with the system in closed loop.

In the project storyline, the experimental team has successfully implanted electrodes and established an interface to the neural system. You can now:
- observe the system,
- estimate hidden structure,
- and actively influence its behaviour through control inputs.

The simulator is provided as a Python package.

The following setup cell installs the package compatible with your operating system and Python version. You only need to run this installation step once.

### Before running the setup

Please ensure that:

- you are using **Python 3.11, 3.12, or 3.13**,
- the `wheels/` folder is present in the same directory as this notebook,
- the wheel files have been unzipped.

Once the package is installed successfully, you can import and use it directly in later cells.

You may delete the installation cell after a successful installation if you wish.

In [1]:
import sys
import platform
import subprocess
import pathlib

ROOT = pathlib.Path(".")
WHEEL_ROOT = ROOT / "wheels"

system = platform.system()
machine = platform.machine().lower()

# -------- 1. Check Python Version --------
supported_versions = {(3, 11), (3, 12), (3, 13)}
py_ver = sys.version_info[:2]

if py_ver not in supported_versions:
    raise RuntimeError(
        f"Unsupported Python version: {py_ver[0]}.{py_ver[1]}.\n"
        "This notebook only supports Python 3.11, 3.12, or 3.13."
    )

py_tag = f"cp{py_ver[0]}{py_ver[1]}"

# -------- 2. Find Pattern Beased on Python Version and Platform --------
patterns = []

if system == "Windows":
    if machine in ("amd64", "x86_64"):
        patterns = [
            f"*{py_tag}*win_amd64.whl",
        ]
    elif machine in ("x86", "i386", "i686"):
        patterns = [
            f"*{py_tag}*win32.whl",
        ]
    else:
        raise RuntimeError(
            f"Unsupported Windows architecture: {machine}.\n"
            "Expected amd64/x86_64 or x86/i386/i686."
        )

elif system == "Darwin":
    # Prioritize universal2, then specific architectures
    if machine in ("arm64", "aarch64"):
        patterns = [
            f"*{py_tag}*macosx*universal2.whl",
            f"*{py_tag}*macosx*arm64.whl",
        ]
    elif machine in ("x86_64", "amd64"):
        patterns = [
            f"*{py_tag}*macosx*universal2.whl",
            f"*{py_tag}*macosx*x86_64.whl",
        ]
    else:
        raise RuntimeError(
            f"Unsupported macOS architecture: {machine}.\n"
            "Expected x86_64/amd64 or arm64/aarch64."
        )

elif system == "Linux":
    # Prioritize manylinux, then musllinux, for each architecture
    if machine in ("x86_64", "amd64"):
        patterns = [
            f"*{py_tag}*manylinux*x86_64.whl",
            f"*{py_tag}*musllinux*x86_64.whl",
        ]
    elif machine in ("i686", "x86"):
        patterns = [
            f"*{py_tag}*manylinux*i686.whl",
            f"*{py_tag}*musllinux*i686.whl",
        ]
    elif machine in ("arm64", "aarch64"):
        patterns = [
            f"*{py_tag}*manylinux*aarch64.whl",
            f"*{py_tag}*musllinux*aarch64.whl",
        ]
    else:
        raise RuntimeError(
            f"Unsupported Linux architecture: {machine}."
        )

else:
    raise RuntimeError(
        f"Unsupported operating system: {system}."
    )

# -------- 3. Find wheel --------
wheel = None
for pattern in patterns:
    matches = sorted(WHEEL_ROOT.rglob(pattern))
    if matches:
        wheel = matches[0]
        break

if wheel is None:
    raise RuntimeError(
        f"No compatible wheel found for:\n"
        f"  Python tag: {py_tag}\n"
        f"  System: {system}\n"
        f"  Architecture: {machine}\n\n"
        f"Searched under: {WHEEL_ROOT}\n"
        f"Patterns tried:\n  " + "\n  ".join(patterns)
    )

# -------- 4. Install --------
print(f"Installing wheel: {wheel.name}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", str(wheel)
])
print("Package installed.")

Installing wheel: gg4-1.2.5-cp312-cp312-win_amd64.whl
Package installed.


## Using the provided simulator

The simulator is provided through the `GG4` package. The main object you will work with is:

- `brain = Brain(random_seed=...)`

You can think of the simulator as a dynamical system that evolves over time:
- you observe noisy neural activity,
- optionally apply stimulation inputs,
- and study how the system responds.

The simulator exposes only a limited interface. You do **not** have direct access to the internal state of the system.

Useful methods and attributes:

- `brain.input_dim` — required length of the command vector `u(t)`
- `brain.next_state(u)` — advance the brain by one time step using input `u`
- `brain.next_state()` — advance one step with no input
- `brain.measure()` — read the noisy observation `y(t)`
- `brain.current_time_stamp` — current time index

### Important note about inputs
The elements of `u(t)` are clipped internally to the range `[0, 1]`.
That means:
- negative values will not help,
- very large values will not have additional effect once clipped,
- it is sensible to think about **input scaling** and **control effort**.

The following code provides an example of how to interact with the simulator.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from GG4 import Brain

# Create a new brain instance
brain = Brain(random_seed=0)

# Input dimension of the brain (storyline: the number of electrodes implanted in the brain for exciting)
U = brain.input_dim
print("Brain input dimension =", U)

# There are three available interfaces to interact with the brain:
## 1. 'brain.current_time_stamp': int, 
#   the current timepoint in the experiment. It can only be the rvalue (has no setter).
print("\nMethod 1: brain.current_time_stamp")
print("current_time_stamp =", brain.current_time_stamp)  # should be 0 at the beginning of the experiment

## 2. 'brain.measure': Callable[void, List[float]], 
#   measure the current neural activity. The output is a 1D array of shape (Neurons,).
print("\nMethod 2: brain.measure()")
measurement = np.array(brain.measure())  # convert list to numpy array for better analysis
print("shape of measurement =", measurement.shape)  # should be (Neurons,)

## 3. 'brain.next_state': Callable[Optional[none|List[float]], void], 
#   apply the input to the brain and move to the next timepoint. The input should be a 1D array of shape (U,).
#   If the input is None, it means no stimulation is applied to the brain at this timepoint.
print("\nMethod 3: brain.next_state(input)")
print("current time point: ", brain.current_time_stamp)  # should be 0
brain.next_state()  # empty input, move to the next timepoint
print("current time point: ", brain.current_time_stamp)  # should be 1
input_to_apply = np.random.rand(U)  # random input for testing
brain.next_state(input_to_apply)  # apply input and move to the next timepoint
print("current time point: ", brain.current_time_stamp)  # should be 2

Brain input dimension = 2

Method 1: brain.current_time_stamp
current_time_stamp = 0

Method 2: brain.measure()
shape of measurement = (16,)

Method 3: brain.next_state(input)
current time point:  0
current time point:  1
current time point:  2
